In [1]:
import sys
print(sys.executable)

/home/b5ng/.conda/envs/ClinicalDigitalTwin/bin/python


In [2]:
import os
import wfdb
import pandas as pd

In [12]:
BASE_RECORDS_DIR = "/home/b5ng/teams/b03/ecg/files"
ECG_CSV_PATH     = "/home/b5ng/private/ClinicalDigitalTwin/data/processed/ecg_data.csv"   # your preprocessed ECG metadata CSV
N_RECORDS_TEST   = 3 

In [13]:
print("=" * 60)
print("TEST 1: Base records directory")
print("=" * 60)

if os.path.exists(BASE_RECORDS_DIR):
    print(f"✓ Directory exists: {BASE_RECORDS_DIR}")
    contents = os.listdir(BASE_RECORDS_DIR)
    print(f"  Contents (first 5): {contents[:5]}")
else:
    print(f"✗ Directory NOT found: {BASE_RECORDS_DIR}")
    print("  Check your base_records_dir path.")
    exit(1)

TEST 1: Base records directory
✓ Directory exists: /home/b5ng/teams/b03/ecg/files
  Contents (first 5): ['p1079', 'p1097', 'p1103', 'p1174', 'p1864']


In [16]:
print("\n" + "=" * 60)
print("TEST 2: ECG metadata CSV")
print("=" * 60)

if not os.path.exists(ECG_CSV_PATH):
    print(f"✗ CSV not found: {ECG_CSV_PATH}")
    exit(1)

df = pd.read_csv(ECG_CSV_PATH)
df = df[df['in_ed']==1]
print(f"✓ Loaded {ECG_CSV_PATH}: {len(df)} rows, {len(df.columns)} columns")

if "path" not in df.columns:
    print(f"✗ No 'path' column found. Columns: {list(df.columns)}")
    exit(1)

print(f"✓ 'path' column present")
print(f"  Sample paths:")
for p in df["path"].head(3):
    print(f"    {p}")


TEST 2: ECG metadata CSV
✓ Loaded /home/b5ng/private/ClinicalDigitalTwin/data/processed/ecg_data.csv: 158284 rows, 26 columns
✓ 'path' column present
  Sample paths:
    files/p1659/p16598616/s40000035/40000035
    files/p1837/p18370366/s40000084/40000084
    files/p1257/p12576058/s40000115/40000115


In [18]:
print("\n" + "=" * 60)
print(f"TEST 3: Reading {N_RECORDS_TEST} WFDB records")
print("=" * 60)

n_ok     = 0
n_failed = 0

for i, raw_path in enumerate(df["path"].head(N_RECORDS_TEST)):
    # Mirror the same path resolution logic used in xgboost_embedding.py
    p = os.path.splitext(raw_path)[0]
    if p.startswith("files/"):
        p = p[len("files/"):]
    record_path = os.path.join(BASE_RECORDS_DIR, p)

    print(f"\n  Record {i+1}: {record_path}")

    try:
        rec = wfdb.rdrecord(record_path)
        sig = rec.p_signal.T  # (channels, time)
        print(f"  ✓ Read OK — shape: {sig.shape}, fs: {rec.fs} Hz, leads: {rec.sig_name}")
        n_ok += 1
    except Exception as e:
        print(f"  ✗ Failed: {e}")
        n_failed += 1


TEST 3: Reading 3 WFDB records

  Record 1: /home/b5ng/teams/b03/ecg/files/p1659/p16598616/s40000035/40000035
  ✓ Read OK — shape: (12, 5000), fs: 500 Hz, leads: ['I', 'II', 'III', 'aVR', 'aVF', 'aVL', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

  Record 2: /home/b5ng/teams/b03/ecg/files/p1837/p18370366/s40000084/40000084
  ✓ Read OK — shape: (12, 5000), fs: 500 Hz, leads: ['I', 'II', 'III', 'aVR', 'aVF', 'aVL', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

  Record 3: /home/b5ng/teams/b03/ecg/files/p1257/p12576058/s40000115/40000115
  ✓ Read OK — shape: (12, 5000), fs: 500 Hz, leads: ['I', 'II', 'III', 'aVR', 'aVF', 'aVL', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
